In [7]:
%pip install pandas

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [8]:
import pandas as pd

In [9]:
ANIME_CSV_PATH = 'mal_anime.csv'
CHAR_CSV_PATH = 'mal_characters_detailed.csv'

In [10]:
df_anime = pd.read_csv(ANIME_CSV_PATH)
df_characters = pd.read_csv(CHAR_CSV_PATH)

# Preprocess Anime

In [11]:
df_anime.head()

,myanimelist_id,title,description,image,Type,Episodes,Status,Premiered,Released_Season,Released_Year,...,Demographic,Duration,Rating,Score,Ranked,Popularity,Members,Favorites,characters,source_url
0,1,Cowboy Bebop,"Crime is timeless. By the year 2071, humanity ...",https://cdn.myanimelist.net/images/anime/4/196...,TV,26,Finished Airing,Spring 1998,Spring,1998.0,...,NaN,24 min. per ep.,R - 17+ (violence & profanity),8.75,#48,#42,"2,008,019","87,916","[{""id"": 3, ""name"": ""Black, Jet"", ""url"": ""https...",https://myanimelist.net/anime/1/Cowboy_Bebop
1,5,Cowboy Bebop: Tengoku no Tobira,"Another day, another bounty—such is the life o...",https://cdn.myanimelist.net/images/anime/1439/...,Movie,1.0,Finished Airing,NaN,NaN,NaN,...,NaN,1 hr. 55 min.,R - 17+ (violence & profanity),8.38,#232,#649,"403,604","1,748","[{""id"": 3, ""name"": ""Black, Jet"", ""url"": ""https...",https://myanimelist.net/anime/5/Cowboy_Bebop__...
2,6,Trigun,"Vash the Stampede is the man with a $$60,000,0...",https://cdn.myanimelist.net/images/anime/1130/...,TV,26,Finished Airing,Spring 1998,Spring,1998.0,...,Shounen,24 min. per ep.,PG-13 - Teens 13 or older,8.22,#385,#265,"815,140","17,193","[{""id"": 713, ""name"": ""Stryfe, Meryl"", ""url"": ""...",https://myanimelist.net/anime/6/Trigun
3,7,Witch Hunter Robin,"Though hidden away from the general public, Wi...",https://cdn.myanimelist.net/images/anime/10/19...,TV,26.0,Finished Airing,Summer 2002,Summer,2002.0,...,NaN,25 min. per ep.,PG-13 - Teens 13 or older,7.23,#3344,#1979,"125,868",686,"[{""id"": 300, ""name"": ""Amon"", ""url"": ""https://m...",https://myanimelist.net/anime/7/Witch_Hunter_R...
4,8,Bouken Ou Beet,It is the dark century and the people are suff...,https://cdn.myanimelist.net/images/anime/7/215...,TV,52,Finished Airing,Fall 2004,Fall,2004.0,...,Shounen,23 min. per ep.,PG - Children,6.92,#4887,#5765,"16,456",18,"[{""id"": 9054, ""name"": ""Beet"", ""url"": ""https://...",https://myanimelist.net/anime/8/Bouken_Ou_Beet


In [12]:
df_characters = pd.read_csv(CHAR_CSV_PATH)

In [13]:
import json

def parse_character_attributes(attributes_str):
    parsed_attrs = {}
    if pd.notna(attributes_str):
        try:
            attributes_list = json.loads(attributes_str)
            if isinstance(attributes_list, list):
                for attr_dict in attributes_list:
                    if isinstance(attr_dict, dict) and 'name' in attr_dict and 'value' in attr_dict:
                        parsed_attrs[attr_dict['name']] = attr_dict['value']
        except json.JSONDecodeError:
            # Malformed JSON, return empty dictionary
            return {}
    return parsed_attrs

df_characters['parsed_attributes_dict'] = df_characters['attributes'].apply(parse_character_attributes)

print("New column 'parsed_attributes_dict' created with extracted attributes.")
display(df_characters[['full_name', 'attributes', 'parsed_attributes_dict']].head())

New column 'parsed_attributes_dict' created with extracted attributes.


,full_name,attributes,parsed_attributes_dict
0,Spike Spiegel,"[{""name"": ""Birthdate"", ""value"": ""June 26, 2044...","{'Birthdate': 'June 26, 2044', 'Height': '185 ..."
1,Faye Valentine,"[{""name"": ""Birthday"", ""value"": ""August 14, 199...","{'Birthday': 'August 14, 1994'}"
2,Jet Black,NaN,{}
3,Ein,NaN,{}
4,Ichigo Kurosaki,"[{""name"": ""Race"", ""value"": ""Human Quincy, Holl...","{'Race': 'Human Quincy, Hollow', 'Birthday': '..."


In [14]:
false_positive = ['Age', 'Background', 'Personality', 'Comments', 'Known relatives', 'Famous quote', 'Subjects she is good at', 'Ability parameters (out of 5)', 'Ability Parameters (out of 5):', '(Anime) Drive to Duel', 'Experience with playing musical instruments', "Unique Skill - Demon Wolf's KingResistance", 'Things he doesn’t like', 
                  'Would Like to Hide', 'Hobby and Special Skills', 'How he spends days off', 'Favorite type of girl', 'Spends his allowance on', 'Most visited spot in the school',
                  'Most frequented spot in school', 'Experience with Playing Musical Instruments', 'Special skill aside from tennis', 'Pet she wants to raise', 
                  'What he wants most at the moment', 'Rikkaidai Fuzoku Middle School', 'Favorite type of books', "Doesn't like/bad at doing", "Things I'm not good at", "Way to pass time when alone"]
for index, row in df_characters.iterrows():
    attributes_dict = row['parsed_attributes_dict']

    if pd.notna(attributes_dict):
        try:
            # Iterate over a copy of the dictionary items to avoid RuntimeError
            for key, value in list(attributes_dict.items()):
                if key in false_positive:
                  continue
                full_str = key + ": " + value
                if (len(key.split()) > 3) and (len(full_str.split())>9):
                  print(full_str)
                  print()
                  # Ensure description is a string before concatenation
                  current_description = df_characters.loc[index, 'description']
                  if pd.isna(current_description):
                      df_characters.loc[index, 'description'] = full_str
                  else:
                      df_characters.loc[index, 'description'] = full_str + current_description
                  df_characters.loc[index, 'parsed_attributes_dict'].pop(key)
        except json.JSONDecodeError:
            # Skip entries that are not valid JSON
            continue

A youkai who is Sesshoumaru's servant: not especially powerful but extremely loyal, he is usually a comical character. Sesshoumaru typically responds to Jaken's failures or more annoying behavior with a sound beating, but Jaken believes this is Sesshomaru's right and never wavers in his loyalty. Jaken started following him in the first place after being captivated by his power and majesty when he slayed a threatening demon, saving Jaken's life (this was incidental; Sesshomaru killed the demon simply because it was in his way. Note: these events were related ONLY in the anime). ("Jaken" means "meanness" or "unkindness".)

The main human character of the series: a pink-haired, red-eyed, 16-year-old, high school student obsessed with mysteries and the occult. Despite her love of strange cases, she never manages to find out who Loki is-- partly due to the fact that she has absolutely no sixth sense, despite being the daughter of a Shinto priest.

It's all in the name with Vicious: he is ru

In [15]:
# character_attributes_df = pd.json_normalize(df_characters['parsed_attributes_dict'])
# character_attributes_df['character_id'] = df_characters['character_id']

# print("First few rows of character_attributes_df:")
# display(character_attributes_df.head())

# print("Info of character_attributes_df:")
# character_attributes_df.info()

In [16]:
# character_attributes_df.to_csv('character_attributes_df.csv', index=False)
# print("df_anime has been saved to 'df_anime_processed.csv'")

In [17]:
from collections import Counter

# Collect all attribute keys from the parsed_attributes_dict column
all_keys = []
for attr_dict in df_characters['parsed_attributes_dict']:
    if isinstance(attr_dict, dict):
        all_keys.extend(attr_dict.keys())

# Count the frequency of each attribute key
key_counts = Counter(all_keys)

# Get the 10 most common attributes
top_attributes = key_counts.most_common(30)

print("Top 10 most used attributes:")
for attribute, count in top_attributes:
    print(f"{attribute}: {count}")

Top 10 most used attributes:
Age: 5919
Height: 5375
Birthday: 4594
Weight: 2775
Blood type: 1810
Blood Type: 1224
Episode: 914
Gender: 871
Likes: 768
Occupation: 725
Race: 622
Gender Identity: 604
Affiliation: 598
Dislikes: 567
Birthdate: 554
Debut: 528
Position: 468
Class: 427
Hobbies: 407
BWH: 310
Zodiac: 307
Eye Color: 305
Hair Color: 284
Birthplace: 264
Nationality: 253
Eye color: 252
(Source: 235
Species: 225
Hair color: 223
Horoscope: 221


In [18]:
# Function to clean attribute keys
def clean_attribute_key(key):
    """Clean attribute keys by removing special characters and normalizing case"""
    if not isinstance(key, str):
        return key
    # Remove leading/trailing whitespace and special characters like *
    cleaned = key.strip().lstrip('*(#-•➣・').strip()
    # Normalize to title case for consistency
    cleaned = cleaned.lower()
    return cleaned

# Apply cleaning to all parsed_attributes_dict
for index, row in df_characters.iterrows():
    attributes_dict = row['parsed_attributes_dict']
    
    if isinstance(attributes_dict, dict) and attributes_dict:
        # Create a new dictionary with cleaned keys
        cleaned_dict = {}
        for key, value in attributes_dict.items():
            cleaned_key = clean_attribute_key(key)
            # If the cleaned key already exists, keep the first occurrence
            if cleaned_key not in cleaned_dict:
                cleaned_dict[cleaned_key] = value
        
        # Update the dataframe with cleaned dictionary
        df_characters.at[index, 'parsed_attributes_dict'] = cleaned_dict

print("Attribute keys have been cleaned and normalized.")
print("\nRecounting attributes after cleaning:")

# Recount attributes with cleaned keys
all_keys_cleaned = []
for attr_dict in df_characters['parsed_attributes_dict']:
    if isinstance(attr_dict, dict):
        all_keys_cleaned.extend(attr_dict.keys())

key_counts_cleaned = Counter(all_keys_cleaned)
top_attributes_cleaned = key_counts_cleaned.most_common(20)

print("Top 30 most used attributes after cleaning:")
for attribute, count in top_attributes_cleaned:
    print(f"{attribute}: {count}")
    print(f"\n{attribute.upper()} (appears {count} times):")
    examples_found = 0
    for i, row in df_characters.iterrows():
        attr_dict = row['parsed_attributes_dict']
        if isinstance(attr_dict, dict) and attribute in attr_dict:
            print(f"  - {row['full_name']}: {attr_dict[attribute]}")
            examples_found += 1
            if examples_found >= 10:  # Show 3 examples per attribute
                break

Attribute keys have been cleaned and normalized.

Recounting attributes after cleaning:
Top 30 most used attributes after cleaning:
age: 5953

AGE (appears 5953 times):
  - Ichigo Kurosaki: 15 (beginning); 17 (currently)
  - Edward Elric: 15-16 (series), 18 (movie, end of the series)
  - Alphonse Elric: 14-15 (10 at the 2003 end, 13 at movie, 17 at the manga/fmab end)
  - Sasuke Uchiha: 12-13 (I); 16-17 (II); 19 (The Last: Naruto the Movie); 32 (Boruto: Naruto the Movie)
  - Itachi Uchiha: 17-18 (I); 21 (II)
  - Naruto Uzumaki: 12-13 (Naruto part I), 15-17 (part II), 19 (The Last: Naruto the Movie), 27 (Naruto epilogue), 32 (Boruto: Naruto the Movie)
  - Haruhi Fujioka: 16
  - Tamaki Suou: 17
  - Mitsukuni Haninozuka: 17
  - Takashi Morinozuka: 18
height: 5407

HEIGHT (appears 5407 times):
  - Spike Spiegel: 185 cm (6' 1")
  - Ichigo Kurosaki: 174->181 cm
  - Rukia Kuchiki: 144 cm (4' 8")
  - Orihime Inoue: 5'2" (157 cm)
  - Sasuke Uchiha: 153.2 cm(I); 168 cm (II); 175 cm (The Last: Na

In [19]:
top_attributes_cleaned

[('age', 5953),
 ('height', 5407),
 ('birthday', 4624),
 ('blood type', 3060),
 ('weight', 2802),
 ('episode', 915),
 ('gender', 881),
 ('likes', 769),
 ('occupation', 726),
 ('race', 626),
 ('gender identity', 604),
 ('affiliation', 599),
 ('dislikes', 567),
 ('eye color', 561),
 ('birthdate', 554),
 ('debut', 528),
 ('hair color', 509),
 ('position', 490),
 ('class', 451),
 ('hobbies', 407)]

In [31]:
# Create a new dataframe with character_id and top attributes as columns
# Define the attributes we want to extract based on the top attributes
top_attribute_keys = [attr for attr, count in top_attributes_cleaned]

# Function to convert attribute names to snake_case
def to_snake_case(text):
    """Convert text to snake_case"""
    if not isinstance(text, str):
        return text
    # Replace spaces and hyphens with underscores
    snake = text.replace(' ', '_').replace('-', '_')
    # Remove any special characters except underscores
    snake = ''.join(c if c.isalnum() or c == '_' else '' for c in snake)
    # Replace multiple underscores with single underscore
    while '__' in snake:
        snake = snake.replace('__', '_')
    # Remove leading/trailing underscores
    snake = snake.strip('_')
    return snake.lower()

# Initialize a list to store the data
character_attributes_data = []

for index, row in df_characters.iterrows():
    character_id = row['character_id']
    attributes_dict = row['parsed_attributes_dict']
    
    # Create a dictionary for this character with character_id and all top attributes
    char_data = {'character_id': character_id}
    
    # Add each top attribute as a column with snake_case name
    if isinstance(attributes_dict, dict):
        for attr_key in top_attribute_keys:
            snake_case_key = to_snake_case(attr_key)
            char_data[snake_case_key] = attributes_dict.get(attr_key, None)
    else:
        for attr_key in top_attribute_keys:
            snake_case_key = to_snake_case(attr_key)
            char_data[snake_case_key] = None
    
    character_attributes_data.append(char_data)

# Create the dataframe
df_character_attributes = pd.DataFrame(character_attributes_data)

print(f"Created dataframe with {len(df_character_attributes)} characters and {len(df_character_attributes.columns)} columns")
print(f"\nColumns: {list(df_character_attributes.columns)}")
print(f"\nFirst few rows:")
display(df_character_attributes.head(10))

# Show some statistics
print(f"\nNon-null counts for each attribute:")
print(df_character_attributes.count())

Created dataframe with 139497 characters and 21 columns

Columns: ['character_id', 'age', 'height', 'birthday', 'blood_type', 'weight', 'episode', 'gender', 'likes', 'occupation', 'race', 'gender_identity', 'affiliation', 'dislikes', 'eye_color', 'birthdate', 'debut', 'hair_color', 'position', 'class', 'hobbies']

First few rows:


,character_id,age,height,birthday,blood_type,weight,episode,gender,likes,occupation,...,gender_identity,affiliation,dislikes,eye_color,birthdate,debut,hair_color,position,class,hobbies
0,1,None,"185 cm (6' 1"")",None,O,70 kg (155 lbs),None,None,None,None,...,None,None,None,None,"June 26, 2044",None,None,None,None,None
1,2,None,None,"August 14, 1994",None,None,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None
2,3,None,None,None,None,None,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None
3,4,None,None,None,None,None,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None
4,5,15 (beginning); 17 (currently),174->181 cm,July 15 (Cancer),None,61->66 kg,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None
5,6,None,"144 cm (4' 8"")",January 14 (Capricorn),None,33 kg (72.6 lb),None,None,None,"Shinigami in the 13th Division, Lieutenant of...",...,None,None,None,None,None,None,None,None,None,None
6,7,None,"5'2"" (157 cm)",September 3,B,45 kg,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None
7,11,"15-16 (series), 18 (movie, end of the series)",None,1899,None,None,None,None,None,State Alchemist,...,None,None,None,None,None,None,None,None,None,None
8,12,"14-15 (10 at the 2003 end, 13 at movie, 17 at ...",None,None,None,None,None,None,None,Alchemist,...,None,None,None,None,None,None,None,None,None,None
9,13,12-13 (I); 16-17 (II); 19 (The Last: Naruto th...,153.2 cm(I); 168 cm (II); 175 cm (The Last: Na...,July 23,AB,43.5kg(I); 52.2kg (II); 55kg (The Last: Naruto...,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None



Non-null counts for each attribute:
character_id       139497
age                  5953
height               5407
birthday             4624
blood_type           3060
weight               2802
episode               915
gender                881
likes                 769
occupation            726
race                  626
gender_identity       604
affiliation           599
dislikes              567
eye_color             561
birthdate             554
debut                 528
hair_color            509
position              490
class                 451
hobbies               407
dtype: int64


In [33]:
# Drop rows where all attribute columns are null (keeping only character_id)
attribute_columns = [col for col in df_character_attributes.columns if col != 'character_id']
df_character_attributes = df_character_attributes.dropna(subset=attribute_columns, how='all')

print(f"After dropping characters with no attributes: {len(df_character_attributes)} characters remaining")
display(df_character_attributes.head(10))

After dropping characters with no attributes: 10734 characters remaining


,character_id,age,height,birthday,blood_type,weight,episode,gender,likes,occupation,...,gender_identity,affiliation,dislikes,eye_color,birthdate,debut,hair_color,position,class,hobbies
0,1,None,"185 cm (6' 1"")",None,O,70 kg (155 lbs),None,None,None,None,...,None,None,None,None,"June 26, 2044",None,None,None,None,None
1,2,None,None,"August 14, 1994",None,None,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None
4,5,15 (beginning); 17 (currently),174->181 cm,July 15 (Cancer),None,61->66 kg,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None
5,6,None,"144 cm (4' 8"")",January 14 (Capricorn),None,33 kg (72.6 lb),None,None,None,"Shinigami in the 13th Division, Lieutenant of...",...,None,None,None,None,None,None,None,None,None,None
6,7,None,"5'2"" (157 cm)",September 3,B,45 kg,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None
7,11,"15-16 (series), 18 (movie, end of the series)",None,1899,None,None,None,None,None,State Alchemist,...,None,None,None,None,None,None,None,None,None,None
8,12,"14-15 (10 at the 2003 end, 13 at movie, 17 at ...",None,None,None,None,None,None,None,Alchemist,...,None,None,None,None,None,None,None,None,None,None
9,13,12-13 (I); 16-17 (II); 19 (The Last: Naruto th...,153.2 cm(I); 168 cm (II); 175 cm (The Last: Na...,July 23,AB,43.5kg(I); 52.2kg (II); 55kg (The Last: Naruto...,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None
10,14,17-18 (I); 21 (II),175 cm (I); 178 cm (II),June 9th (Gemini),AB,57 kg (I); 58 kg (II),None,None,None,None,...,None,Akatsuki,None,None,None,None,None,None,None,None
11,15,None,"164 cm (5'5"")",None,None,None,None,Male,None,Professional Boxer; Makunouchi Fishing Boat,...,None,None,None,None,None,None,None,None,None,None


In [34]:
# Compare gender and gender_identity columns
if 'gender' in df_character_attributes.columns and 'gender_identity' in df_character_attributes.columns:
    # Count different scenarios
    both_same = 0
    both_different = 0
    gender_has_value_identity_none = 0
    gender_none_identity_has_value = 0
    both_none = 0
    
    for index, row in df_character_attributes.iterrows():
        gender = row['gender']
        gender_identity = row['gender_identity']
        
        # Both are None
        if pd.isna(gender) and pd.isna(gender_identity):
            both_none += 1
        # Both have values
        elif pd.notna(gender) and pd.notna(gender_identity):
            if gender == gender_identity:
                both_same += 1
            else:
                both_different += 1
        # Only gender has value
        elif pd.notna(gender) and pd.isna(gender_identity):
            gender_has_value_identity_none += 1
        # Only gender_identity has value
        elif pd.isna(gender) and pd.notna(gender_identity):
            gender_none_identity_has_value += 1
    
    print("=== Comparison between 'gender' and 'gender_identity' ===\n")
    print(f"Both have same value: {both_same}")
    print(f"Both have values but different: {both_different}")
    print(f"Only 'gender' has value (gender_identity is None): {gender_has_value_identity_none}")
    print(f"Only 'gender_identity' has value (gender is None): {gender_none_identity_has_value}")
    print(f"Both are None: {both_none}")
    print(f"\nTotal: {both_same + both_different + gender_has_value_identity_none + gender_none_identity_has_value + both_none}")
    
    # Show examples of different values
    if both_different > 0:
        print("\n=== Examples where both have values but are different ===")
        count = 0
        for index, row in df_character_attributes.iterrows():
            if pd.notna(row['gender']) and pd.notna(row['gender_identity']) and row['gender'] != row['gender_identity']:
                char_id = row['character_id']
                char_name = df_characters[df_characters['character_id'] == char_id]['full_name'].values[0] if len(df_characters[df_characters['character_id'] == char_id]) > 0 else 'Unknown'
                print(f"  - {char_name}: gender='{row['gender']}', gender_identity='{row['gender_identity']}'")
                count += 1
                if count >= 10:
                    break
else:
    print("One or both columns ('gender', 'gender_identity') do not exist in the dataframe")
    print(f"Available columns: {list(df_character_attributes.columns)}")

=== Comparison between 'gender' and 'gender_identity' ===

Both have same value: 0
Both have values but different: 0
Only 'gender' has value (gender_identity is None): 881
Only 'gender_identity' has value (gender is None): 604
Both are None: 9249

Total: 10734


In [35]:
# Combine gender_identity into gender column
# If gender is None but gender_identity has a value, use gender_identity
if 'gender' in df_character_attributes.columns and 'gender_identity' in df_character_attributes.columns:
    for index, row in df_character_attributes.iterrows():
        if pd.isna(row['gender']) and pd.notna(row['gender_identity']):
            df_character_attributes.at[index, 'gender'] = row['gender_identity']
    
    # Drop the gender_identity column since we've merged it into gender
    df_character_attributes = df_character_attributes.drop(columns=['gender_identity'])
    
    print("Successfully combined 'gender_identity' into 'gender' column")
    print(f"Dropped 'gender_identity' column")
    print(f"\nUpdated columns: {list(df_character_attributes.columns)}")
    print(f"\nGender column non-null count: {df_character_attributes['gender'].notna().sum()}")
else:
    print("One or both columns do not exist")

Successfully combined 'gender_identity' into 'gender' column
Dropped 'gender_identity' column

Updated columns: ['character_id', 'age', 'height', 'birthday', 'blood_type', 'weight', 'episode', 'gender', 'likes', 'occupation', 'race', 'affiliation', 'dislikes', 'eye_color', 'birthdate', 'debut', 'hair_color', 'position', 'class', 'hobbies']

Gender column non-null count: 1485


In [36]:
# Show unique values of gender column
if 'gender' in df_character_attributes.columns:
    unique_genders = df_character_attributes['gender'].unique()
    gender_counts = df_character_attributes['gender'].value_counts(dropna=False)
    
    print(f"Unique gender values: {len(unique_genders)}")
    print(f"\nUnique values:")
    for gender in unique_genders:
        print(f"  - {gender}")
    
    print(f"\nValue counts:")
    print(gender_counts)
else:
    print("'gender' column does not exist in the dataframe")

Unique gender values: 25

Unique values:
  - None
  - Male
  - Female
  - male
  - female
  - male  (can change to female)
  - F
  - Female (brought up as male)
  - None
  - Unknown
  - Male (Normal)/Female (Kämpfer)
  - FEMALE
  - Feminine
  - Bird
  - Genderless
  - Male(+Female)
  - unknown
  - Possesses the ability to switch between a male or female body at will.
  - Male (Half-Demon)
  - Female (Purebred Demon (Devil))
  - Female (Succubus)
  - Female (Demi-beastfolk)
  - Female (Dragonfolk)
  - Intersex
  - Non-Binary

Value counts:
gender
None                                                                      9249
Male                                                                       876
Female                                                                     531
male                                                                        38
female                                                                      14
Intersex                                             

In [37]:
# Clean gender column - normalize to 'Male' and 'Female'
if 'gender' in df_character_attributes.columns:
    # Create a mapping for normalization
    def normalize_gender(gender_value):
        if pd.isna(gender_value):
            return gender_value
        
        gender_str = str(gender_value).strip().lower()
        
        # Normalize female variants
        if gender_str in ['female', 'f']:
            return 'Female'
        # Normalize male variants
        elif gender_str in ['male', 'm']:
            return 'Male'
        # Keep other values as-is for now
        else:
            return gender_value
    
    # Apply normalization
    df_character_attributes['gender'] = df_character_attributes['gender'].apply(normalize_gender)
    
    print("Gender column has been normalized")
    print(f"\nUpdated gender value counts:")
    print(df_character_attributes['gender'].value_counts(dropna=False))
    
    # Find characters with gender values other than 'Male' or 'Female'
    other_genders = df_character_attributes[
        (df_character_attributes['gender'].notna()) & 
        (~df_character_attributes['gender'].isin(['Male', 'Female']))
    ]
    
    if len(other_genders) > 0:
        print(f"\n=== Characters with gender other than 'Male' or 'Female': {len(other_genders)} ===")
        for index, row in other_genders.iterrows():
            char_id = row['character_id']
            gender = row['gender']
            char_name = df_characters[df_characters['character_id'] == char_id]['full_name'].values[0] if len(df_characters[df_characters['character_id'] == char_id]) > 0 else 'Unknown'
            print(f"  Character ID: {char_id}, Name: {char_name}, Gender: '{gender}'")
    else:
        print("\nAll characters have gender values of 'Male', 'Female', or None")
else:
    print("'gender' column does not exist in the dataframe")

Gender column has been normalized

Updated gender value counts:
gender
None                                                                      9249
Male                                                                       914
Female                                                                     547
Intersex                                                                     5
Unknown                                                                      2
Non-Binary                                                                   2
None                                                                         1
Male (Normal)/Female (Kämpfer)                                               1
Feminine                                                                     1
Bird                                                                         1
Female (brought up as male)                                                  1
Male(+Female)                                               

In [38]:
df_character_attributes.to_csv('character_attributes_cleaned.csv', index=False)

In [39]:
# Check what columns we actually have in df_character_attributes
print("Available columns in df_character_attributes:")
print(list(df_character_attributes.columns))
print(f"\nTotal columns: {len(df_character_attributes.columns)}")

Available columns in df_character_attributes:
['character_id', 'age', 'height', 'birthday', 'blood_type', 'weight', 'episode', 'gender', 'likes', 'occupation', 'race', 'affiliation', 'dislikes', 'eye_color', 'birthdate', 'debut', 'hair_color', 'position', 'class', 'hobbies']

Total columns: 20


In [40]:
%pip install neo4j python-dotenv

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [43]:
from neo4j import GraphDatabase
import os
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

URI = os.getenv("NEO4J_URI", "bolt://localhost:7687")
USER = os.getenv("NEO4J_USER", "neo4j")
PASSWORD = os.getenv("NEO4J_PASSWORD", "your_password")

def get_driver():
    return GraphDatabase.driver(URI, auth=(USER, PASSWORD))

# Function to update character attributes in Neo4j
def update_character_attributes_batch(driver, data_batch, attribute_columns):
    if not data_batch:
        return
    
    # Build the SET clause dynamically based on available attributes
    # Convert snake_case to camelCase for Neo4j properties
    def snake_to_camel(snake_str):
        components = snake_str.split('_')
        return components[0] + ''.join(x.title() for x in components[1:])
    
    # Build SET statements for each attribute
    set_statements = []
    for col in attribute_columns:
        camel_case_col = snake_to_camel(col)
        set_statements.append(f"c.{camel_case_col} = row.{col}")
    
    set_clause = ',\n        '.join(set_statements)
    
    cypher_query = f"""
    UNWIND $batch AS row
    MATCH (c:Character {{malCharacterId: row.character_id}})
    SET {set_clause}
    """
    
    with driver.session() as session:
        session.run(cypher_query, batch=data_batch)

print("Neo4j connection setup complete")

Neo4j connection setup complete


In [44]:
# Prepare data for Neo4j ingestion
# Convert dataframe to list of dictionaries, converting NaN to None
character_data_for_neo4j = []

# Get all attribute columns (exclude character_id)
attribute_cols = [col for col in df_character_attributes.columns if col != 'character_id']

for index, row in df_character_attributes.iterrows():
    char_dict = {'character_id': int(row['character_id'])}
    
    # Dynamically add all attribute columns
    for col in attribute_cols:
        char_dict[col] = row[col] if pd.notna(row[col]) else None
    
    character_data_for_neo4j.append(char_dict)

print(f"Prepared {len(character_data_for_neo4j)} character records for Neo4j ingestion")
print(f"Attributes to be updated: {attribute_cols}")
print(f"\nFirst record example:")
print(character_data_for_neo4j[0])

Prepared 10734 character records for Neo4j ingestion
Attributes to be updated: ['age', 'height', 'birthday', 'blood_type', 'weight', 'episode', 'gender', 'likes', 'occupation', 'race', 'affiliation', 'dislikes', 'eye_color', 'birthdate', 'debut', 'hair_color', 'position', 'class', 'hobbies']

First record example:
{'character_id': 1, 'age': None, 'height': '185 cm (6\' 1")', 'birthday': None, 'blood_type': 'O', 'weight': '70 kg (155 lbs)', 'episode': None, 'gender': None, 'likes': None, 'occupation': None, 'race': None, 'affiliation': None, 'dislikes': None, 'eye_color': None, 'birthdate': 'June 26, 2044', 'debut': None, 'hair_color': None, 'position': None, 'class': None, 'hobbies': None}


In [46]:
# Ingest character attributes to Neo4j
def main():
    driver = get_driver()
    
    # Get attribute columns (exclude character_id)
    attribute_cols = [col for col in df_character_attributes.columns if col != 'character_id']
    
    print("Starting character attributes ingestion to Neo4j...")
    print(f"Attributes to update: {', '.join(attribute_cols)}")
    
    total = len(character_data_for_neo4j)
    BATCH_SIZE = 500  # Process in batches of 500
    
    for i in range(0, total, BATCH_SIZE):
        batch = character_data_for_neo4j[i : i + BATCH_SIZE]
        
        # Update Neo4j
        update_character_attributes_batch(driver, batch, attribute_cols)
        
        print(f"Progress: {min(i + BATCH_SIZE, total)}/{total} characters processed...")
    
    print("\n✅ Character attributes ingestion complete!")
    driver.close()

# Run the ingestion
main()

Starting character attributes ingestion to Neo4j...
Attributes to update: age, height, birthday, blood_type, weight, episode, gender, likes, occupation, race, affiliation, dislikes, eye_color, birthdate, debut, hair_color, position, class, hobbies
Progress: 500/10734 characters processed...
Progress: 500/10734 characters processed...
Progress: 1000/10734 characters processed...
Progress: 1000/10734 characters processed...
Progress: 1500/10734 characters processed...
Progress: 1500/10734 characters processed...
Progress: 2000/10734 characters processed...
Progress: 2000/10734 characters processed...
Progress: 2500/10734 characters processed...
Progress: 2500/10734 characters processed...
Progress: 3000/10734 characters processed...
Progress: 3000/10734 characters processed...
Progress: 3500/10734 characters processed...
Progress: 3500/10734 characters processed...
Progress: 4000/10734 characters processed...
Progress: 4000/10734 characters processed...
Progress: 4500/10734 characters pr